# PART A — Start with the Data

This section identifies TicketWise's data requirements, verifies the selected CloudJoi PDF sources, and inspects the extracted document content before data preparation.


### Cell 1 — Workspace setup

In [4]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

# Define the main project and data directories
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Create the raw and processed data folders
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

# Display the workspace directories
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


### Cell 3 — Verify uploaded inputs

In [5]:
# Cell 3 — Verify uploaded inputs
from pathlib import Path

# Define the raw data directory and project paths
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Set compatibility aliases for notebook examples
BASE = BASE_DIR
OUT = PROCESSED_DIR

# List all uploaded files in the raw data folder
files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

# Confirm that at least one input file is available
assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 3
cloudjoi_faq.pdf bytes: 287340
cloudjoi_refund_policy.pdf bytes: 114073
cloudjoi_ticketing_terms.pdf bytes: 382254
Upload verified. Select your relevant input(s) for Step A3.


### Step A3 — Initial PDF inspection

In [6]:
# Cell A3 — Initial PDF inspection

from pathlib import Path

# Import PDF reader
try:
    from pypdf import PdfReader
except ImportError:
    !pip install pypdf
    from pypdf import PdfReader

# Locate the raw PDF files
pdf_files = [
    RAW_DIR / "cloudjoi_faq.pdf",
    RAW_DIR / "cloudjoi_ticketing_terms.pdf",
    RAW_DIR / "cloudjoi_refund_policy.pdf"
]

for pdf_path in pdf_files:
    print("=" * 70)
    print("FILE:", pdf_path.name)

    reader = PdfReader(str(pdf_path))

    print("Number of pages:", len(reader.pages))

    # Extract text from the first page for initial inspection
    first_page_text = reader.pages[0].extract_text() or ""

    print("First page extracted characters:", len(first_page_text))
    print("\nFIRST PAGE TEXT PREVIEW:")
    print(first_page_text[:1000])

    if not first_page_text.strip():
        print("WARNING: No readable text extracted from the first page.")

    print("\n")

FILE: cloudjoi_faq.pdf
Number of pages: 2
First page extracted characters: 3903

FIRST PAGE TEXT PREVIEW:
Frequently Asked Questions
Discover everything you need to know about buying tickets, streaming live and
on-demand shows, and enjoying unforgettable experiences with CloudJoi – your
ultimate ticketing, streaming, and community platform in Malaysia!
Ticketing Process:
Is there a time limit when purchasing tickets online?
All seats are reserved for 15 minutes. If the booking is not finalized within this
time frame, the seats will be released for the next potential buyer.
Why am I encountering a credit card error at checkout?
If you've reached the purchase page, entered your card number, and
submitted your order, but received a credit card error message, we suggest
double-checking your card details and available funds, then attempting the
transaction again.
Why is there a ticket limit when purchasing?
For general events, we allow a maximum of 12 tickets per order. However,
ticket limi

### Step A3 — Check text extraction across all PDF pages

In [7]:
# Cell A3 — Check text extraction across all PDF pages

for pdf_path in pdf_files:
    reader = PdfReader(str(pdf_path))

    print("=" * 60)
    print("FILE:", pdf_path.name)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        print(
            f"Page {page_number}: "
            f"{len(text)} extracted characters"
        )

        if not text.strip():
            print("WARNING: No readable text on this page.")

    print()

FILE: cloudjoi_faq.pdf
Page 1: 3903 extracted characters
Page 2: 2274 extracted characters

FILE: cloudjoi_ticketing_terms.pdf
Page 1: 4353 extracted characters
Page 2: 4156 extracted characters
Page 3: 4749 extracted characters
Page 4: 4501 extracted characters
Page 5: 468 extracted characters

FILE: cloudjoi_refund_policy.pdf
Page 1: 611 extracted characters



### Cell 4 — Common metadata helpers

In [8]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source file before processing.
# This prevents accidental selection of the wrong file.
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected

    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]

# Generate a SHA-256 hash to identify the source file.
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

# Save pipeline metadata and lineage records in a JSON file.
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirm that the helper functions are ready.
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


### Cell 4A — Common lineage registry

In [9]:
# Cell 4A — REQUIRED: Initialise Common Lineage Registry

from pathlib import Path

# Store the relationship between source files and prepared outputs
lineage_entries = {}

# Register the source file, prepared output and processing method
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    # Check that the source file exists
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )

    # Check that the prepared output exists and is not empty
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )

    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )

    # Ensure a processing description is provided
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )

    processing = processing.strip()

    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )

    # Ensure the source and output belong to the correct project folders
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    # Update an existing source-output pair instead of duplicating it
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }

    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )

# Confirm that the lineage registry is ready
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


# PART B — Prepare the Data

## Pathway A — PDF/TXT Document Preparation

The following cells implement the PDF/TXT preparation pathway selected for TicketWise. The preparation process includes text extraction, cleaning, structure-aware transformation, validation and prepared-data export.


### Step 5A1 — PDF/TXT Extraction, Cleaning and Preparation

In [10]:
# Cell 5A1.1 — Extract PDF Text and Compare Chunking Approaches
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source PDF files
pdf_files = [
    select_source(relative_path="cloudjoi_faq.pdf"),
    select_source(relative_path="cloudjoi_ticketing_terms.pdf"),
    select_source(relative_path="cloudjoi_refund_policy.pdf")
]

# Step 2: Define fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")

    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]

# Step 3: Define paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")

    parts = []
    current = ""

    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()

        if not segment:
            continue

        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""

            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment

        else:
            if current:
                parts.append(current)

            current = segment

    if current:
        parts.append(current)

    return parts

# Step 4: Extract and clean text from all selected PDFs
# Store each document's chunks separately to preserve source identity.
document_chunks = {}

for f in pdf_files:
    reader = PdfReader(str(f))

    # Extract text from every page
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]

    # Combine extracted text and clean unnecessary spaces and tabs
    text = "\n".join(content for _, content in pages)
    clean = re.sub(r"[ \t]+", " ", text).strip()

    print("=" * 70)
    print("Source:", f.name)
    print("Pages/segments:", len(pages))
    print("Extracted characters:", len(text))
    print("Cleaned characters:", len(clean))
    print("Content sample:", repr(clean[:300]))

    assert clean, f"No usable text found in {f.name}. Check extraction or OCR needs."

    # Step 5: Generate both chunking approaches
    fixed = fixed_chunks(clean, size=500)
    structured = paragraph_chunks(clean, size=500)

    # Store the results for later comparison and export
    document_chunks[f.name] = {
        "source": f,
        "pages": pages,
        "text": clean,
        "fixed": fixed,
        "structured": structured
    }

    # Step 6: Display chunking comparison
    print("\n--- Chunking Comparison ---")
    print("Fixed chunks:", len(fixed))
    print("Paragraph-aware chunks:", len(structured))

    print("\nFixed sample:")
    print(repr(fixed[0][:180]))

    print("\nParagraph-aware sample:")
    print(repr(structured[0][:180]))

    print("\n")

print("PATHWAY A1 completed successfully.")


Source: cloudjoi_faq.pdf
Pages/segments: 2
Extracted characters: 6178
Cleaned characters: 6178
Content sample: 'Frequently Asked Questions\nDiscover everything you need to know about buying tickets, streaming live and\non-demand shows, and enjoying unforgettable experiences with CloudJoi – your\nultimate ticketing, streaming, and community platform in Malaysia!\nTicketing Process:\nIs there a time limit when purch'

--- Chunking Comparison ---
Fixed chunks: 13
Paragraph-aware chunks: 13

Fixed sample:
'Frequently Asked Questions\nDiscover everything you need to know about buying tickets, streaming live and\non-demand shows, and enjoying unforgettable experiences with CloudJoi – you'

Paragraph-aware sample:
'Frequently Asked Questions\nDiscover everything you need to know about buying tickets, streaming live and\non-demand shows, and enjoying unforgettable experiences with CloudJoi – you'


Source: cloudjoi_ticketing_terms.pdf
Pages/segments: 5
Extracted characters: 18231
Cleaned chara

In [11]:
# Cell 5A1.2 — Inspect Chunking Differences

# Compare the two chunking approaches for each document
for filename, data in document_chunks.items():
    fixed = data["fixed"]
    structured = data["structured"]
    clean = data["text"]

    print("=" * 70)
    print("DOCUMENT:", filename)

    # Check whether both approaches produced identical chunks
    print("Identical chunk lists:", fixed == structured)

    # Check how many paragraph separators exist in the extracted text
    paragraph_breaks = len(re.findall(r"\n\s*\n", clean))
    print("Detected paragraph separators:", paragraph_breaks)

    # Compare the lengths of the first three chunks
    print("Fixed chunk lengths:", [len(c) for c in fixed[:3]])
    print("Paragraph-aware chunk lengths:", [len(c) for c in structured[:3]])

    # Inspect the end of the first chunk from each approach
    print("\nFixed chunk ending:")
    print(repr(fixed[0][-120:]))

    print("\nParagraph-aware chunk ending:")
    print(repr(structured[0][-120:]))

    print()


DOCUMENT: cloudjoi_faq.pdf
Identical chunk lists: False
Detected paragraph separators: 1
Fixed chunk lengths: [500, 500, 500]
Paragraph-aware chunk lengths: [500, 500, 500]

Fixed chunk ending:
'ot finalized within this\ntime frame, the seats will be released for the next potential buyer.\nWhy am I encountering a cr'

Paragraph-aware chunk ending:
'ot finalized within this\ntime frame, the seats will be released for the next potential buyer.\nWhy am I encountering a cr'

DOCUMENT: cloudjoi_ticketing_terms.pdf
Identical chunk lists: False
Detected paragraph separators: 1
Fixed chunk lengths: [500, 500, 500]
Paragraph-aware chunk lengths: [500, 500, 500]

Fixed chunk ending:
'rms and conditions set forth herein.\nThis ticketing terms & conditions page outlines the terms and conditions\ngoverning '

Paragraph-aware chunk ending:
'rms and conditions set forth herein.\nThis ticketing terms & conditions page outlines the terms and conditions\ngoverning '

DOCUMENT: cloudjoi_refund_policy.pdf

In [12]:
# Cell 5A1.3 — Examine the First Differing Chunks

for filename, data in document_chunks.items():
    fixed = data["fixed"]
    structured = data["structured"]

    print("=" * 70)
    print("DOCUMENT:", filename)

    # Find the first chunk that differs between the methods
    for i, (fixed_chunk, structured_chunk) in enumerate(
        zip(fixed, structured)
    ):
        if fixed_chunk != structured_chunk:
            print("First differing chunk:", i + 1)

            print("\nFixed-size chunk ending:")
            print(repr(fixed_chunk[-200:]))

            print("\nParagraph-aware chunk ending:")
            print(repr(structured_chunk[-200:]))

            break
    else:
        print("No differences found.")

    print()


DOCUMENT: cloudjoi_faq.pdf
First differing chunk: 8

Fixed-size chunk ending:
' once the transaction is confirmed, no refunds or exchanges\nSearch event, organiser, venue, or artists\n\nare permitted unless the event is canceled or postponed. However,\ndepending on the event organiz'

Paragraph-aware chunk ending:
'ed\nbefore proceeding to payment.\nI cannot attend the event. Can I receive a refund?\nUnfortunately, once the transaction is confirmed, no refunds or exchanges\nSearch event, organiser, venue, or artists'

DOCUMENT: cloudjoi_ticketing_terms.pdf
First differing chunk: 9

Fixed-size chunk ending:
"laints or\nSearch event, organiser, venue, or artists\n\nexchange requests resulting from the ticket purchaser's failure to attend. It should\nbe noted that we do not guarantee the delivery of pre-event n"

Paragraph-aware chunk ending:
't\npurchaser is responsible for ensuring attendance. The notification serves as a\ngoodwill gesture, and we explicitly disclaim liability for any c

In [13]:
# Cell 5A1.4 — Inspect FAQ Question Structure

faq_data = document_chunks["cloudjoi_faq.pdf"]
faq_text = faq_data["text"]

# Display lines that appear to be questions
question_lines = [
    line.strip()
    for line in faq_text.splitlines()
    if line.strip().endswith("?")
]

print("Potential questions found:", len(question_lines))
print("\nQuestions:")

for i, question in enumerate(question_lines, 1):
    print(f"{i}. {question}")


Potential questions found: 21

Questions:
1. Is there a time limit when purchasing tickets online?
2. Why am I encountering a credit card error at checkout?
3. Why is there a ticket limit when purchasing?
4. Why am I seeing 'Fully Queued'? Are the tickets still available?
5. What payment methods does CloudJoi offer?
6. do?
7. I accidentally deleted the email, how do I retrieve the tickets?
8. purchase history, what should I do?
9. Where do I download the mobile apps?
10. Do I need to print out the e-tickets?
11. What is CloudJoi's refund and exchange policy?
12. request a rebate?
13. I cannot attend the event. Can I receive a refund?
14. Where can I watch the show?
15. Can I be late?
16. How do I watch it in full screen mode?
17. How fast do I need my internet to experience CloudTheatre?
18. Can I share my Passcode to my friend?
19. Can I still visit the show afterward?
20. Do I need to watch the show immediately after purchasing the ticket?
21. How do the 24/48 hours work?


In [14]:
# Cell 5A1.5 — Inspect FAQ Question and Answer Boundaries

faq_lines = document_chunks["cloudjoi_faq.pdf"]["text"].splitlines()

# Inspect the ticket retrieval and refund sections
for i, line in enumerate(faq_lines):
    if any(keyword in line.lower() for keyword in [
        "ticket retrieval",
        "i purchased a ticket",
        "i accidentally deleted",
        "i tried to login",
        "refunds, exchanges",
        "i forgot to apply",
        "i cannot attend"
    ]):
        print(f"\n--- Around line {i + 1} ---")

        for j in range(max(0, i - 1), min(len(faq_lines), i + 6)):
            print(f"{j + 1:03d}: {repr(faq_lines[j])}")



--- Around line 30 ---
029: 'Banking as well as Malaysia and International Credit Cards.'
030: 'Ticket Retrieval and Assistance:'
031: "I purchased a ticket but didn't receive a confirmation email. What should I"
032: 'do?'
033: "Please check your email inbox, junk, and promotions folders. If you can't find"
034: 'it, please contact us\xa0for assistance.'
035: 'I accidentally deleted the email, how do I retrieve the tickets?'

--- Around line 31 ---
030: 'Ticket Retrieval and Assistance:'
031: "I purchased a ticket but didn't receive a confirmation email. What should I"
032: 'do?'
033: "Please check your email inbox, junk, and promotions folders. If you can't find"
034: 'it, please contact us\xa0for assistance.'
035: 'I accidentally deleted the email, how do I retrieve the tickets?'
036: "If you have a CloudJoi ID, simply log in using the same email address. You'll"

--- Around line 35 ---
034: 'it, please contact us\xa0for assistance.'
035: 'I accidentally deleted the email, how do I

In [15]:
# Cell 5A1.6 — Extract FAQ Question-and-Answer Records

import re

# Step 1: Prepare the extracted FAQ text
faq_lines = [
    line.replace("\xa0", " ").strip()
    for line in document_chunks["cloudjoi_faq.pdf"]["text"].splitlines()
]

# Step 2: Define document categories and navigation text
categories = {
    "Ticketing Process:",
    "Payment Methods:",
    "Ticket Retrieval and Assistance:",
    "Refunds, Exchanges, and Rebates:",
    "Watching Online",
    "Watch On-Demand"
}

navigation_items = {
    "Organiser Dashboard",
    "Find My Tickets",
    "Frequently Asked Questions",
    "Ticketing Terms & Conditions",
    "Brand Assets",
    "About Us",
    "Contact Us",
    "Sell Tickets with Us",
    "Blog",
    "Careers",
    "Privacy Policy Terms of Use",
    "RESOURCES",
    "ABOUT",
    "CONNECT WITH US",
    "DOWNLOAD THE APP"
}

question_start = re.compile(
    r"^(Is |Why |What |I |Where |Do |Can |How |My )",
    re.IGNORECASE
)

# Step 3: Preserve document-level information
faq_title = "Frequently Asked Questions"
faq_intro = " ".join(faq_lines[1:4])

# Step 4: Set up FAQ record storage
faq_records = []
current_category = None
current_question = None
current_answer = []
question_complete = False

def save_faq_record():
    if current_question and current_answer:
        faq_records.append({
            "category": current_category,
            "question": current_question.strip(),
            "answer": " ".join(current_answer).strip()
        })

# Step 5: Process each line
for line in faq_lines:
    if not line:
        continue

    if line == "Organiser Dashboard":
        break

    if line == "Search event, organiser, venue, or artists":
        continue

    if line in categories:
        save_faq_record()
        current_category = line.rstrip(":")
        current_question = None
        current_answer = []
        question_complete = False
        continue

    if line in navigation_items:
        continue

    if current_category is None:
        continue

    starts_question = bool(question_start.match(line))

    # Start a new question after the previous question has ended
    if starts_question and question_complete:
        save_faq_record()
        current_question = line
        current_answer = []
        question_complete = False

    # Continue collecting the question if it spans multiple lines
    elif current_question and not question_complete:
        current_question += " " + line

    # Start a new question
    elif current_question is None:
        if starts_question:
            current_question = line

    # Collect the answer after the question is complete
    elif question_complete:
        current_answer.append(line)

    # Check whether the question has reached its ending
    if current_question and not question_complete:
        if line.endswith("?") or line == "My chat box kept loading without any content.":
            question_complete = True

# Step 6: Save the final record
save_faq_record()

# Step 7: Display extraction results
print("FAQ title:", faq_title)
print("\nIntroduction:", faq_intro)
print("\nTotal FAQ records extracted:", len(faq_records))

for i, record in enumerate(faq_records, 1):
    print("\n" + "=" * 70)
    print(f"FAQ {i} | Category: {record['category']}")
    print("Question:", record["question"])
    print("Answer:", record["answer"])

# Step 8: Validate the results
assert len(faq_records) == 22, (
    f"Expected 22 FAQ records, found {len(faq_records)}"
)

assert all(
    record["question"] and record["answer"]
    for record in faq_records
), "Incomplete FAQ record detected."

print("\nFAQ extraction completed successfully.")


FAQ title: Frequently Asked Questions

Introduction: Discover everything you need to know about buying tickets, streaming live and on-demand shows, and enjoying unforgettable experiences with CloudJoi – your ultimate ticketing, streaming, and community platform in Malaysia!

Total FAQ records extracted: 22

FAQ 1 | Category: Ticketing Process
Question: Is there a time limit when purchasing tickets online?
Answer: All seats are reserved for 15 minutes. If the booking is not finalized within this time frame, the seats will be released for the next potential buyer.

FAQ 2 | Category: Ticketing Process
Question: Why am I encountering a credit card error at checkout?
Answer: If you've reached the purchase page, entered your card number, and submitted your order, but received a credit card error message, we suggest double-checking your card details and available funds, then attempting the transaction again.

FAQ 3 | Category: Ticketing Process
Question: Why is there a ticket limit when purch

In [16]:
# Cell 5A1.7 — Identify Ticketing Terms Section Headings

terms_text = document_chunks["cloudjoi_ticketing_terms.pdf"]["text"]

# Display lines that may be section headings
for line in terms_text.splitlines():
    line = line.strip()

    if line and len(line) < 60:
        print(line)


Ticketing Terms & Conditions
Conditions
you must not make a purchase.
General Information
dealt with by the event organiser.
or any other causes beyond its control.
capacity without prior notice.
check with the respective organiser.
General Ticket Purchase
Tickets are non-refundable, all ticket purchases are final.
age and understands and accepts these conditions of use.
prices or discounts.
Promo Code Usage
specified validity period.
and discounts after the completion of the checkout process.
codes before finalizing their purchase.
Exchange of Tickets
Postponement section for further guidance.
Search event, organiser, venue, or artists
Admission
citizens.
disorderly or unbecoming.
reasons.
guaranteed and no refund request would be entertained.
substances and recording devices.
Streaming
used to stream on one device only.
connection for smooth streaming.
Personal Information
facilitate ticket purchase or other services available.
shall be entitled to use and process such data.
Data Col

In [17]:
# Cell 5A1.8 — Define Ticketing Terms Section Headings

terms_headings = [
    "General Information",
    "General Ticket Purchase",
    "Promo Code Usage",
    "Exchange of Tickets",
    "Admission",
    "Streaming",
    "Personal Information",
    "Data Collection and Usage",
    "Refunds",
    "Prohibited Conduct",
    "Limitation of Liability",
    "Donations via CloudJoi",
    "Other Disclaimers",
    "Governing Law",
    "Queries, Compliants and Dispute",
    "Amendments to Terms"
]

# Confirm the identified headings
print("Total section headings:", len(terms_headings))

for i, heading in enumerate(terms_headings, 1):
    print(f"{i}. {heading}")


Total section headings: 16
1. General Information
2. General Ticket Purchase
3. Promo Code Usage
4. Exchange of Tickets
5. Admission
6. Streaming
7. Personal Information
8. Data Collection and Usage
9. Refunds
10. Prohibited Conduct
11. Limitation of Liability
12. Donations via CloudJoi
13. Other Disclaimers
14. Governing Law
15. Queries, Compliants and Dispute
16. Amendments to Terms


In [18]:
# Cell 5A1.9 — Include the Ticketing Terms Introduction

terms_headings = ["Introduction"] + terms_headings

print("Total sections:", len(terms_headings))

for i, heading in enumerate(terms_headings, 1):
    print(f"{i}. {heading}")


Total sections: 17
1. Introduction
2. General Information
3. General Ticket Purchase
4. Promo Code Usage
5. Exchange of Tickets
6. Admission
7. Streaming
8. Personal Information
9. Data Collection and Usage
10. Refunds
11. Prohibited Conduct
12. Limitation of Liability
13. Donations via CloudJoi
14. Other Disclaimers
15. Governing Law
16. Queries, Compliants and Dispute
17. Amendments to Terms


In [19]:
# Cell 5A1.10 — Inspect Ticketing Terms Introduction

print(repr(terms_text[:2500]))


'Ticketing Terms & Conditions\nSale of CloudJoi Tickets are subject to the following Terms and\nConditions\nCloudTheatre Sdn Bhd (operating under the brand name "CloudJoi", referred to\nas "we", "us", or "our") is a private limited company incorporated in Malaysia, with\nbusiness registration number 202001024125. By purchasing a ticket, you\nacknowledge and agree to be bound by the terms and conditions set forth herein.\nThis ticketing terms & conditions page outlines the terms and conditions\ngoverning your purchase of tickets for events hosted on our platform.\nBy purchasing one or more Items from us, you acknowledge that you have read,\nunderstood and agree to be bound by the terms and conditions of this ticketing\nterms and conditions. If you do not agree with this Purchase Policy or any other\napplicable terms and conditions, or if you cannot comply with any of them, then\nyou must not make a purchase.\nWe reserve the right from time to time to make changes to this Ticketing Terms

In [20]:
# Cell 5A1.11 — Extract Ticketing Terms by Section

# Remove the document title and unwanted website navigation
lines = terms_text.splitlines()
lines = [
    line.strip()
    for line in lines
    if line.strip()
]

# Remove the document title
if lines and lines[0] == "Ticketing Terms & Conditions":
    lines = lines[1:]

# Remove website navigation and footer content
if "Organiser Dashboard" in lines:
    lines = lines[:lines.index("Organiser Dashboard")]

# Remove an unwanted website search phrase
lines = [
    line for line in lines
    if line != "Search event, organiser, venue, or artists"
]

# Identify the actual section headings
section_headings = terms_headings[1:]

# Separate the introduction and named sections
terms_sections = {}
current_section = "Introduction"
terms_sections[current_section] = []

for line in lines:
    if line in section_headings:
        current_section = line
        terms_sections[current_section] = []
    else:
        terms_sections[current_section].append(line)

# Join extracted lines while retaining their line breaks
terms_sections = {
    heading: "\n".join(content).strip()
    for heading, content in terms_sections.items()
}

# Display section counts and content lengths
print("Sections extracted:", len(terms_sections))

for heading, content in terms_sections.items():
    print(f"{heading}: {len(content)} characters")

print("\n--- Introduction Preview ---")
print(terms_sections["Introduction"])

print("\n--- General Information Preview ---")
print(terms_sections["General Information"][:500])


Sections extracted: 17
Introduction: 1350 characters
General Information: 589 characters
General Ticket Purchase: 803 characters
Promo Code Usage: 948 characters
Exchange of Tickets: 663 characters
Admission: 839 characters
Streaming: 293 characters
Personal Information: 391 characters
Data Collection and Usage: 1116 characters
Refunds: 1362 characters
Prohibited Conduct: 774 characters
Limitation of Liability: 3465 characters
Donations via CloudJoi: 704 characters
Other Disclaimers: 2000 characters
Governing Law: 159 characters
Queries, Compliants and Dispute: 1555 characters
Amendments to Terms: 322 characters

--- Introduction Preview ---
Sale of CloudJoi Tickets are subject to the following Terms and
Conditions
CloudTheatre Sdn Bhd (operating under the brand name "CloudJoi", referred to
as "we", "us", or "our") is a private limited company incorporated in Malaysia, with
business registration number 202001024125. By purchasing a ticket, you
acknowledge and agree to be bound by the t

In [21]:
# Cell 5A1.12 — Inspect Extracted Terms Sections

for heading in ["Limitation of Liability", "Other Disclaimers"]:
    print(f"\n--- {heading} ---")
    print(terms_sections[heading])


--- Limitation of Liability ---
To the fullest extent allowed by law, we (including our affiliates, parent
companies, subsidiaries, and their officers, directors, employees, agents, legal
representatives, and subcontractors) and the pertinent Event Organizer shall not
be held responsible for any loss, injury, or damage to any individual (including
yourself) or property, regardless of the cause (whether by our actions and/or
those of the Event Organizer):
in situations where there is no violation of contract or legal duty of care on
our part or the relevant Event Organiser;
in cases where the loss or damage is not directly attributable to such
violation (except for death or personal injury arising from our or an Event
Partner's negligence);
or to the extent that any escalation in such loss or damage stems from your
negligence or a breach of any terms outlined in this Purchase Policy, and/or
any other relevant terms and conditions, and/or any applicable laws or
regulations.
To the fulle

In [22]:
# Cell 5A1.13 — Inspect Paragraph Boundaries in Terms Sections

for heading in ["Limitation of Liability", "Other Disclaimers"]:
    print(f"\n--- {heading} ---")

    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", terms_sections[heading])
        if p.strip()
    ]

    print("Paragraphs detected:", len(paragraphs))

    for i, paragraph in enumerate(paragraphs, 1):
        print(f"\nParagraph {i}:")
        print(paragraph[:250])


--- Limitation of Liability ---
Paragraphs detected: 1

Paragraph 1:
To the fullest extent allowed by law, we (including our affiliates, parent
companies, subsidiaries, and their officers, directors, employees, agents, legal
representatives, and subcontractors) and the pertinent Event Organizer shall not
be held respo

--- Other Disclaimers ---
Paragraphs detected: 1

Paragraph 1:
The ticket holder acknowledges and assumes all risks associated with attending
the event, including weather changes, travel to and from the venue, and
collisions. The ticket holder releases CloudJoi, Venue Owner, Event Organiser,
event performers/pla


In [23]:
# Cell 5A1.14 — Create Section-Aware Terms Records

terms_records = []

for i, (heading, content) in enumerate(terms_sections.items(), 1):
    if not content.strip():
        continue

    terms_records.append({
        "record_id": f"ticketing-terms-{i:03d}",
        "section": heading,
        "text": f"{heading}\n{content}",
        "source": "cloudjoi_ticketing_terms.pdf",
        "processing": "section-aware"
    })

# Check the generated records
print("Total section records:", len(terms_records))

for record in terms_records:
    print(
        record["record_id"],
        "|",
        record["section"],
        "|",
        len(record["text"]),
        "characters"
    )

assert len(terms_records) == 17
assert all(record["text"].strip() for record in terms_records)

print("\nTicketing Terms records: PASS")

Total section records: 17
ticketing-terms-001 | Introduction | 1363 characters
ticketing-terms-002 | General Information | 609 characters
ticketing-terms-003 | General Ticket Purchase | 827 characters
ticketing-terms-004 | Promo Code Usage | 965 characters
ticketing-terms-005 | Exchange of Tickets | 683 characters
ticketing-terms-006 | Admission | 849 characters
ticketing-terms-007 | Streaming | 303 characters
ticketing-terms-008 | Personal Information | 412 characters
ticketing-terms-009 | Data Collection and Usage | 1142 characters
ticketing-terms-010 | Refunds | 1370 characters
ticketing-terms-011 | Prohibited Conduct | 793 characters
ticketing-terms-012 | Limitation of Liability | 3489 characters
ticketing-terms-013 | Donations via CloudJoi | 727 characters
ticketing-terms-014 | Other Disclaimers | 2018 characters
ticketing-terms-015 | Governing Law | 173 characters
ticketing-terms-016 | Queries, Compliants and Dispute | 1587 characters
ticketing-terms-017 | Amendments to Terms | 3

In [24]:
# Cell 5A1.15 — Inspect Refund Policy Text

refund_text = document_chunks["cloudjoi_refund_policy.pdf"]["text"]

print("Characters:", len(refund_text))
print("\nExtracted content:")
print(refund_text)

Characters: 611

Extracted content:
Home Refunds, Cancellations & Changes What is CloudJoi’s refund and exchange policy?
What is CloudJoi’s refund and exchange policy?
Last updated on May 4, 2025
All ticket sales are final. We do not allow refunds, exchanges, or cancellations once
a booking is confirmed—unless the event is officially canceled or postponed.
To avoid any mix-ups, our system provides several confirmation steps before you
complete your purchase. We strongly recommend double-checking your event,
date, time, and ticket details before making payment.
© 2026, All Rights Reserved, CloudTheatre Sdn. Bhd. (1380445-V)
CloudJoi Support


In [25]:
# Cell 5A1.16 — Create Document-Level Refund Policy Record

# Remove website navigation and footer content
refund_clean = refund_text

refund_clean = refund_clean.replace(
    "Home Refunds, Cancellations & Changes ",
    "",
    1
)

refund_clean = refund_clean.replace(
    "© 2026, All Rights Reserved, CloudTheatre Sdn. Bhd. (1380445-V)",
    ""
)

refund_clean = refund_clean.replace(
    "CloudJoi Support",
    ""
)

# Remove the duplicated title
title = "What is CloudJoi’s refund and exchange policy?"

refund_clean = refund_clean.replace(
    title + "\n" + title,
    title,
    1
).strip()

# Create one document-level record
refund_records = [{
    "record_id": "refund-policy-001",
    "section": "Refund and Exchange Policy",
    "text": refund_clean,
    "source": "cloudjoi_refund_policy.pdf",
    "processing": "document-level"
}]

# Validate the record
assert refund_records[0]["text"].strip()
assert title in refund_records[0]["text"]
assert "Last updated on May 4, 2025" in refund_records[0]["text"]

print("Refund policy record created.")
print("Characters:", len(refund_clean))
print("\nPrepared content:")
print(refund_clean)
print("\nValidation: PASS")


Refund policy record created.
Characters: 445

Prepared content:
What is CloudJoi’s refund and exchange policy?
Last updated on May 4, 2025
All ticket sales are final. We do not allow refunds, exchanges, or cancellations once
a booking is confirmed—unless the event is officially canceled or postponed.
To avoid any mix-ups, our system provides several confirmation steps before you
complete your purchase. We strongly recommend double-checking your event,
date, time, and ticket details before making payment.

Validation: PASS


In [26]:
# Cell 5A1.17 — Validate Extracted FAQ Records

print("Total FAQ records:", len(faq_records))

for i, record in enumerate(faq_records[:3], 1):
    print(f"\n--- FAQ Record {i} ---")
    print("Category:", record["category"])
    print("Question:", record["question"])
    print("Answer:", record["answer"])

assert len(faq_records) == 22
assert all(
    record["question"].strip() and record["answer"].strip()
    for record in faq_records
)

print("\nFAQ validation: PASS")


Total FAQ records: 22

--- FAQ Record 1 ---
Category: Ticketing Process
Question: Is there a time limit when purchasing tickets online?
Answer: All seats are reserved for 15 minutes. If the booking is not finalized within this time frame, the seats will be released for the next potential buyer.

--- FAQ Record 2 ---
Category: Ticketing Process
Question: Why am I encountering a credit card error at checkout?
Answer: If you've reached the purchase page, entered your card number, and submitted your order, but received a credit card error message, we suggest double-checking your card details and available funds, then attempting the transaction again.

--- FAQ Record 3 ---
Category: Ticketing Process
Question: Why is there a ticket limit when purchasing?
Answer: For general events, we allow a maximum of 12 tickets per order. However, ticket limits are determined by the event organizers as part of our measures to ensure fairness and provide everyone with an equal opportunity to obtain ticket

In [27]:
# Cell 5A1.18 — Combine Prepared Records and Metadata

# Identify the original source files
faq_file = select_source(relative_path="cloudjoi_faq.pdf")
terms_file = select_source(relative_path="cloudjoi_ticketing_terms.pdf")
refund_file = select_source(relative_path="cloudjoi_refund_policy.pdf")

# Prepare the FAQ introduction as a separate record
faq_intro_record = {
    "record_id": "faq-introduction-001",
    "category": "Introduction",
    "question": None,
    "answer": faq_intro
}

# Combine the FAQ introduction and question-answer records
all_faq_records = [faq_intro_record] + faq_records

# Add record IDs and source metadata to FAQ records
faq_prepared = [
    {
        "record_id": (
            "faq-introduction-001" if i == 0
            else f"faq-{i:03d}"
        ),
        "text": (
            f"Introduction\n{record['answer']}"
            if i == 0
            else (
                f"Category: {record['category']}\n"
                f"Question: {record['question']}\n"
                f"Answer: {record['answer']}"
            )
        ),
        "source": faq_file.name,
        "source_sha256": source_hash(faq_file),
        "processing": "faq-question-answer-extraction",
        "page": None
    }
    for i, record in enumerate(all_faq_records)
]

# Add source metadata to Ticketing Terms records
terms_prepared = [
    {
        **record,
        "source": terms_file.name,
        "source_sha256": source_hash(terms_file),
        "page": None
    }
    for record in terms_records
]

# Add source metadata to the refund policy record
refund_prepared = [
    {
        **record,
        "source_sha256": source_hash(refund_file),
        "page": None
    }
    for record in refund_records
]

# Combine all three sources
prepared_records = (
    faq_prepared +
    terms_prepared +
    refund_prepared
)

# Check record counts and source metadata
print("FAQ records:", len(faq_prepared))
print("Ticketing Terms records:", len(terms_prepared))
print("Refund policy records:", len(refund_prepared))
print("Total prepared records:", len(prepared_records))

assert len(faq_prepared) == 23
assert len(terms_prepared) == 17
assert len(refund_prepared) == 1
assert len(prepared_records) == 41

assert all(
    record["text"].strip()
    and record["source"]
    and record["source_sha256"]
    and record["processing"]
    for record in prepared_records
)

print("\nCombined records and source metadata: PASS")


FAQ records: 23
Ticketing Terms records: 17
Refund policy records: 1
Total prepared records: 41

Combined records and source metadata: PASS


### Step 5A2 — JSONL Export

In [28]:
# Cell 5A2 — Export Prepared Records as JSONL
import json

# Correct the processing description for the FAQ introduction
prepared_records[0]["processing"] = "document-level-context-extraction"

# Check that every record has a unique ID
record_ids = [record["record_id"] for record in prepared_records]
assert len(record_ids) == len(set(record_ids)), "Duplicate record IDs found."

# Check that all records contain text and source metadata
for record in prepared_records:
    assert record["text"].strip()
    assert record["source"]
    assert record["source_sha256"]
    assert record["processing"]

# Define the prepared-data output path
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"

# Write each record as one JSON line
with prepared_path.open("w", encoding="utf-8") as file:
    for record in prepared_records:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

# Read the exported file again to verify it
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

assert len(exported_records) == 41

print("Prepared data exported successfully.")
print("Output:", prepared_path)
print("Total records:", len(exported_records))
print("File size:", prepared_path.stat().st_size, "bytes")
print("Export validation: PASS")


Prepared data exported successfully.
Output: /content/week3_project/data/processed/prepared_text.jsonl
Total records: 41
File size: 34516 bytes
Export validation: PASS


### Step B3 — Validation and Inspection

In [29]:
# B3.1 — Inspect Representative Prepared Records

# Select one representative record from each source
sample_ids = [1, 24, 40]

for index in sample_ids:
    record = prepared_records[index]

    print("\n--- Prepared Record ---")
    print("ID:", record["record_id"])
    print("Source:", record["source"])
    print("Processing:", record["processing"])
    print("SHA-256:", record["source_sha256"])
    print("Text length:", len(record["text"]), "characters")
    print("\nPrepared text:")
    print(record["text"][:1000])



--- Prepared Record ---
ID: faq-001
Source: cloudjoi_faq.pdf
Processing: faq-question-answer-extraction
SHA-256: 177c3487fdb336346ba5d4657ac16e0403927784d829b97f3a68a4d9f7139efc
Text length: 251 characters

Prepared text:
Category: Ticketing Process
Question: Is there a time limit when purchasing tickets online?
Answer: All seats are reserved for 15 minutes. If the booking is not finalized within this time frame, the seats will be released for the next potential buyer.

--- Prepared Record ---
ID: ticketing-terms-002
Source: cloudjoi_ticketing_terms.pdf
Processing: section-aware
SHA-256: 9d6a697f53049531b223e5e71233559b12468b0833bcc2c95aef96cdd6d02ddf
Text length: 609 characters

Prepared text:
General Information
Any feedbacks or complaints regarding the event will be directed to, and
dealt with by the event organiser.
The Organiser reserves the right to postpone, cancel, interrupt, or stop the
Event due to failure of broadcast, technical issues, instructions by authority,
or any oth

In [30]:
# B3.2 — Validate Prepared Dataset

import json
import re

# Load the exported JSONL file
with prepared_path.open("r", encoding="utf-8") as file:
    validation_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

# Check the total number of records
assert len(validation_records) == 41

# Check that all record IDs are unique
record_ids = [record["record_id"] for record in validation_records]
assert len(record_ids) == len(set(record_ids))

# Check required fields and non-empty text
required_fields = [
    "record_id",
    "text",
    "source",
    "source_sha256",
    "processing"
]

for record in validation_records:
    for field in required_fields:
        assert field in record, f"Missing field: {field}"

    assert record["text"].strip(), "Empty text found."

    # Validate SHA-256 format
    assert re.fullmatch(
        r"[a-fA-F0-9]{64}",
        record["source_sha256"]
    ), f"Invalid SHA-256 hash: {record['record_id']}"

# Check that all three source documents are represented
sources = {record["source"] for record in validation_records}

assert sources == {
    "cloudjoi_faq.pdf",
    "cloudjoi_ticketing_terms.pdf",
    "cloudjoi_refund_policy.pdf"
}

# Check the number of records from each source
source_counts = {}

for record in validation_records:
    source = record["source"]
    source_counts[source] = source_counts.get(source, 0) + 1

expected_counts = {
    "cloudjoi_faq.pdf": 23,
    "cloudjoi_ticketing_terms.pdf": 17,
    "cloudjoi_refund_policy.pdf": 1
}

assert source_counts == expected_counts, (
    f"Unexpected source record distribution: {source_counts}"
)

# Display validation results
print("Total records:", len(validation_records))
print("Unique record IDs:", len(record_ids))

print("\nRecords by source:")
for source, count in source_counts.items():
    print(f"{source}: {count}")

print("\nRequired metadata: PASS")
print("Non-empty text: PASS")
print("SHA-256 format: PASS")
print("Source distribution: PASS")
print("Dataset validation: PASS")

Total records: 41
Unique record IDs: 41

Records by source:
cloudjoi_faq.pdf: 23
cloudjoi_ticketing_terms.pdf: 17
cloudjoi_refund_policy.pdf: 1

Required metadata: PASS
Non-empty text: PASS
SHA-256 format: PASS
Source distribution: PASS
Dataset validation: PASS


In [31]:
# B3.3 — Validate Prepared Output File

import json

# Step 1: Check that the prepared file exists
assert prepared_path.is_file(), "Prepared output missing."

# Step 2: Check that the prepared file is not empty
file_size = prepared_path.stat().st_size

assert file_size > 0, "Prepared output is empty."

# Step 3: Read and validate the JSONL file
with prepared_path.open("r", encoding="utf-8") as file:
    exported_lines = [
        line for line in file
        if line.strip()
    ]

# Confirm every line contains valid JSON
exported_records = []

for line_number, line in enumerate(exported_lines, start=1):
    try:
        exported_records.append(json.loads(line))
    except json.JSONDecodeError as error:
        raise ValueError(
            f"Invalid JSON on line {line_number}: {error}"
        )

# Step 4: Check record count
assert len(exported_records) == len(validation_records), (
    "Exported record count does not match validated dataset."
)

# Step 5: Confirm exported content matches B3.2
assert exported_records == validation_records, (
    "Exported records differ from the validated dataset."
)

# Step 6: Display validation results
print("Output:", prepared_path)
print("File size:", file_size, "bytes")
print("JSONL format: PASS")
print("Record count:", len(exported_records))
print("Exported content matches validated dataset: PASS")

print("\nPrepared output validation: PASS")


Output: /content/week3_project/data/processed/prepared_text.jsonl
File size: 34516 bytes
JSONL format: PASS
Record count: 41
Exported content matches validated dataset: PASS

Prepared output validation: PASS


### Step B4.1 — Register Selected Pathway

In [32]:
# B4.1 — Register Source-to-Output Lineage

# Register each original PDF against the prepared JSONL file
for source_file in pdf_files:
    register_lineage(
        source=source_file,
        output=prepared_path,
        processing=(
            "FAQ question-answer extraction"
            if source_file.name == "cloudjoi_faq.pdf"
            else "Section-aware extraction"
            if source_file.name == "cloudjoi_ticketing_terms.pdf"
            else "Document-level extraction"
        )
    )

# Check the registered lineage entries
print("\nTotal lineage entries:", len(lineage_entries))

assert len(lineage_entries) == 3

registered_sources = {
    entry["source"].name
    for entry in lineage_entries.values()
}

expected_sources = {
    "cloudjoi_faq.pdf",
    "cloudjoi_ticketing_terms.pdf",
    "cloudjoi_refund_policy.pdf"
}

assert registered_sources == expected_sources

print("All three source documents registered: PASS")
print("Lineage registration: PASS")


Registered: cloudjoi_faq.pdf -> prepared_text.jsonl
Registered: cloudjoi_ticketing_terms.pdf -> prepared_text.jsonl
Registered: cloudjoi_refund_policy.pdf -> prepared_text.jsonl

Total lineage entries: 3
All three source documents registered: PASS
Lineage registration: PASS


### Cell 6A — Save Manifest

In [33]:

# Cell 6A — Save the Data Lineage Manifest

# Convert registered lineage entries into JSON-compatible records
lineage_records = []

for entry in lineage_entries.values():
    source = entry["source"]
    output = entry["output"]

    lineage_records.append({
        # Source identification
        "source_id": f"cloudjoi-{source.stem.lower().replace('_', '-')}",
        "source_filename": source.name,
        "source": str(source.relative_to(BASE_DIR)),

        # Source and prepared output hashes
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR)),
        "prepared_sha256": source_hash(output),

        # Processing information
        "processing_note": entry["processing"],
        "processing": entry["processing"],

        # Source information and access rights
        "source_version": "Not verified",
        "access_rights": (
            "Publicly accessible official CloudJoi website content; "
            "reuse and redistribution permissions not independently verified"
        ),

        # Current preparation status
        "status": "prepared"
    })

# Save the manifest
manifest = save_manifest(lineage_records)

# Verify the saved manifest
manifest_path = BASE_DIR / "data" / "week3_manifest.json"

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

assert len(saved_manifest["records"]) == 3
assert saved_manifest["pipeline_version"] == "week3-v1"

print("\nManifest records:", len(saved_manifest["records"]))
print("Manifest location:", manifest_path)
print("Manifest validation: PASS")

Saved lineage: /content/week3_project/data/week3_manifest.json

Manifest records: 3
Manifest location: /content/week3_project/data/week3_manifest.json
Manifest validation: PASS


### Cell 6B — Validate Manifest

In [34]:
# Cell 6B — Validate the Saved Data Lineage Manifest

# Step 1: Load the saved manifest
assert manifest_path.exists(), "Manifest file not found."

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

# Step 2: Check manifest information
assert saved_manifest["pipeline_version"] == "week3-v1"
assert len(saved_manifest["records"]) == 3

print("Manifest structure: PASS")

# Step 3: Check required metadata fields
required_fields = [
    "source_id",
    "source_filename",
    "source_sha256",
    "prepared_output",
    "prepared_sha256",
    "processing_note",
    "source_version",
    "access_rights",
    "status"
]

for record in saved_manifest["records"]:
    for field in required_fields:
        assert field in record and record[field], (
            f"Missing or empty field: {field}"
        )

print("Required metadata fields: PASS")

# Step 4: Validate source and prepared output references
for record in saved_manifest["records"]:

    source_path = BASE_DIR / record["source"]
    output_path = BASE_DIR / record["prepared_output"]

    assert source_path.is_file(), (
        f"Source file missing: {source_path}"
    )

    assert output_path.is_file(), (
        f"Prepared output missing: {output_path}"
    )

    # Verify source SHA-256
    actual_source_hash = source_hash(source_path)

    assert actual_source_hash == record["source_sha256"], (
        f"Source hash mismatch: {record['source_filename']}"
    )

    # Verify prepared output SHA-256
    actual_output_hash = source_hash(output_path)

    assert actual_output_hash == record["prepared_sha256"], (
        f"Prepared output hash mismatch: {record['source_filename']}"
    )

    print(f"PASS: {record['source_filename']}")
    print("  Source reference and hash verified.")
    print("  Prepared output reference and hash verified.")

# Step 5: Final validation result
print("\nAll Cell 6B lineage validation checks passed!")


Manifest structure: PASS
Required metadata fields: PASS
PASS: cloudjoi_faq.pdf
  Source reference and hash verified.
  Prepared output reference and hash verified.
PASS: cloudjoi_ticketing_terms.pdf
  Source reference and hash verified.
  Prepared output reference and hash verified.
PASS: cloudjoi_refund_policy.pdf
  Source reference and hash verified.
  Prepared output reference and hash verified.

All Cell 6B lineage validation checks passed!


# PART C — Build and Document the V1 Data Pipeline

This section develops an executable V1 data pipeline for TicketWise using the preparation method selected in Part B. It includes common pipeline utilities, document processing, pipeline execution, repeatability testing and final output verification.


In [35]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json


# 7.1 Define Standard Project Paths

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# 7.2 Define SHA-256 Checksum Function

def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# 7.3 Define Source Resolution Function

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# 7.4 Define Common V1 Pipeline Execution Function

def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    elif kind == "documents" and source.is_dir():
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {".pdf", ".txt"}
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# 7.5 Define Pipeline Registration and Execution Function

def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first


print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")


Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


### Supporting check — Inspect available raw source files

In [36]:
# Supporting Check — Inspect Available Raw Source Files

# Display the files available in the raw data folder
print("Files in raw folder:")
for file in sorted(RAW_DIR.iterdir()):
    if file.is_file():
        print("-", file.name)

Files in raw folder:
- cloudjoi_faq.pdf
- cloudjoi_refund_policy.pdf
- cloudjoi_ticketing_terms.pdf


### Cell 8A — TicketWise PDF Document Pipeline

In [37]:
# Cell 8A — TicketWise PDF Document Pipeline

import re
import json
from pypdf import PdfReader


# 8A.1 Select the Source Document Folder

SOURCE_A = resolve_source(".", folder=True)


# 8A.2 Define the Preparation Configuration Selected in Part B

CONFIG_A = {
    "method": "structure-aware",
    "faq": "introduction + question-answer",
    "terms": "section-aware",
    "refund": "document-level"
}


# 8A.3 Inspect the Input Documents

def inspect_documents(inputs, cfg):
    inspection = []

    for file in inputs:
        reader = PdfReader(str(file))
        page_texts = [
            page.extract_text() or ""
            for page in reader.pages
        ]

        total_chars = sum(len(text) for text in page_texts)

        if total_chars == 0:
            raise ValueError(
                f"No readable text extracted from {file.name}"
            )

        inspection.append({
            "filename": file.name,
            "pages": len(reader.pages),
            "characters": total_chars
        })

    print("Document inspection:")
    for item in inspection:
        print(item)

    return inspection


# 8A.4 Prepare and Transform the Documents

def prepare_documents(inputs, out, cfg):

    # 8A.4.1 Validate the Selected Source Documents

    expected_sources = {
        "cloudjoi_faq.pdf",
        "cloudjoi_ticketing_terms.pdf",
        "cloudjoi_refund_policy.pdf"
    }

    actual_sources = {file.name for file in inputs}

    assert actual_sources == expected_sources, (
        f"Unexpected source documents: {actual_sources}"
    )

    # Extract text from all source documents
    documents = {}

    for file in inputs:
        reader = PdfReader(str(file))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )
        documents[file.name] = text

    # Identify the three source files
    faq_file = next(
        f for f in inputs if f.name == "cloudjoi_faq.pdf"
    )
    terms_file = next(
        f for f in inputs if f.name == "cloudjoi_ticketing_terms.pdf"
    )
    refund_file = next(
        f for f in inputs if f.name == "cloudjoi_refund_policy.pdf"
    )


    # 8A.4.2 Prepare FAQ Question-and-Answer Records

    faq_lines = [
        line.replace("\xa0", " ").strip()
        for line in documents["cloudjoi_faq.pdf"].splitlines()
    ]

    categories = {
        "Ticketing Process:",
        "Payment Methods:",
        "Ticket Retrieval and Assistance:",
        "Refunds, Exchanges, and Rebates:",
        "Watching Online",
        "Watch On-Demand"
    }

    navigation_items = {
        "Organiser Dashboard",
        "Find My Tickets",
        "Frequently Asked Questions",
        "Ticketing Terms & Conditions",
        "Brand Assets",
        "About Us",
        "Contact Us",
        "Sell Tickets with Us",
        "Blog",
        "Careers",
        "Privacy Policy Terms of Use",
        "RESOURCES",
        "ABOUT",
        "CONNECT WITH US",
        "DOWNLOAD THE APP"
    }

    question_start = re.compile(
        r"^(Is |Why |What |I |Where |Do |Can |How |My )",
        re.IGNORECASE
    )

    # Preserve the FAQ introduction
    faq_intro = " ".join(faq_lines[1:4])

    faq_records = []
    current_category = None
    current_question = None
    current_answer = []
    question_complete = False

    def save_faq_record():
        if current_question and current_answer:
            faq_records.append({
                "category": current_category,
                "question": current_question.strip(),
                "answer": " ".join(current_answer).strip()
            })

    # Extract individual FAQ questions and answers
    for line in faq_lines:
        if not line:
            continue

        if line == "Organiser Dashboard":
            break

        if line == "Search event, organiser, venue, or artists":
            continue

        if line in categories:
            save_faq_record()
            current_category = line.rstrip(":")
            current_question = None
            current_answer = []
            question_complete = False
            continue

        if line in navigation_items:
            continue

        if current_category is None:
            continue

        starts_question = bool(question_start.match(line))

        if starts_question and question_complete:
            save_faq_record()
            current_question = line
            current_answer = []
            question_complete = False

        elif current_question and not question_complete:
            current_question += " " + line

        elif current_question is None:
            if starts_question:
                current_question = line

        elif question_complete:
            current_answer.append(line)

        if current_question and not question_complete:
            if (
                line.endswith("?")
                or line == "My chat box kept loading without any content."
            ):
                question_complete = True

    # Save the final FAQ record
    save_faq_record()

    # Validate FAQ extraction
    assert len(faq_records) == 22, (
        f"Expected 22 FAQ records, found {len(faq_records)}"
    )

    # Create the FAQ introduction record
    faq_prepared = [{
        "record_id": "faq-introduction-001",
        "text": f"Introduction\n{faq_intro}",
        "source": faq_file.name,
        "source_sha256": sha256_file(faq_file),
        "processing": "faq-question-answer-extraction",
        "page": None
    }]

    # Add the extracted FAQ question-and-answer records
    for i, record in enumerate(faq_records, 1):
        faq_prepared.append({
            "record_id": f"faq-{i:03d}",
            "text": (
                f"Category: {record['category']}\n"
                f"Question: {record['question']}\n"
                f"Answer: {record['answer']}"
            ),
            "source": faq_file.name,
            "source_sha256": sha256_file(faq_file),
            "processing": "faq-question-answer-extraction",
            "page": None
        })


    # 8A.4.3 Prepare Ticketing Terms by Section

    terms_text = documents["cloudjoi_ticketing_terms.pdf"]

    terms_headings = [
        "General Information",
        "General Ticket Purchase",
        "Promo Code Usage",
        "Exchange of Tickets",
        "Admission",
        "Streaming",
        "Personal Information",
        "Data Collection and Usage",
        "Refunds",
        "Prohibited Conduct",
        "Limitation of Liability",
        "Donations via CloudJoi",
        "Other Disclaimers",
        "Governing Law",
        "Queries, Compliants and Dispute",
        "Amendments to Terms"
    ]

    # Include the introduction as the first section
    terms_headings = ["Introduction"] + terms_headings

    lines = [
        line.strip()
        for line in terms_text.splitlines()
        if line.strip()
    ]

    # Remove document title and website navigation
    if lines and lines[0] == "Ticketing Terms & Conditions":
        lines = lines[1:]

    if "Organiser Dashboard" in lines:
        lines = lines[:lines.index("Organiser Dashboard")]

    lines = [
        line for line in lines
        if line != "Search event, organiser, venue, or artists"
    ]

    # Group text under the corresponding section headings
    section_headings = terms_headings[1:]
    terms_sections = {"Introduction": []}
    current_section = "Introduction"

    for line in lines:
        if line in section_headings:
            current_section = line
            terms_sections[current_section] = []
        else:
            terms_sections[current_section].append(line)

    # Create section-level prepared records
    terms_records = []

    for i, (heading, content) in enumerate(terms_sections.items(), 1):
        content = "\n".join(content).strip()

        if not content:
            continue

        terms_records.append({
            "record_id": f"ticketing-terms-{i:03d}",
            "text": f"{heading}\n{content}",
            "source": terms_file.name,
            "source_sha256": sha256_file(terms_file),
            "processing": "section-aware",
            "page": None
        })

    # Validate Ticketing Terms extraction
    assert len(terms_records) == 17, (
        f"Expected 17 Ticketing Terms sections, "
        f"found {len(terms_records)}"
    )


    # 8A.4.4 Prepare the Refund Policy as a Document-Level Record

    refund_clean = documents["cloudjoi_refund_policy.pdf"]

    # Remove website navigation and footer content
    refund_clean = refund_clean.replace(
        "Home Refunds, Cancellations & Changes ", "", 1
    )

    refund_clean = refund_clean.replace(
        "© 2026, All Rights Reserved, CloudTheatre Sdn. Bhd. (1380445-V)",
        ""
    )

    refund_clean = refund_clean.replace(
        "CloudJoi Support", ""
    )

    # Remove the duplicated document title
    title = "What is CloudJoi’s refund and exchange policy?"

    refund_clean = refund_clean.replace(
        title + "\n" + title, title, 1
    ).strip()

    # Validate the prepared refund policy text
    assert refund_clean
    assert title in refund_clean
    assert "Last updated on May 4, 2025" in refund_clean

    # Create one document-level record
    refund_records = [{
        "record_id": "refund-policy-001",
        "text": refund_clean,
        "source": refund_file.name,
        "source_sha256": sha256_file(refund_file),
        "processing": "document-level",
        "page": None
    }]


    # 8A.4.5 Combine and Export Prepared Records

    prepared_records = (
        faq_prepared
        + terms_records
        + refund_records
    )

    # Validate the combined dataset
    assert len(prepared_records) == 41

    assert len({
        record["record_id"] for record in prepared_records
    }) == 41

    assert all(
        record["text"].strip()
        for record in prepared_records
    )

    # Export prepared records as JSONL
    output_path = out / "prepared_text.jsonl"

    with output_path.open("w", encoding="utf-8") as f:
        for record in prepared_records:
            f.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Record source-to-output mappings
    mappings = [
        {
            "source": str(file),
            "output": str(output_path)
        }
        for file in inputs
    ]

    method = (
        "Structure-aware preparation: FAQ question-answer "
        "extraction, Ticketing Terms section-aware extraction, "
        "and refund policy document-level extraction."
    )

    print("\nPreparation completed.")
    print("FAQ records:", len(faq_prepared))
    print("Ticketing Terms records:", len(terms_records))
    print("Refund policy records:", len(refund_records))
    print("Total prepared records:", len(prepared_records))
    print("Output:", output_path)

    return [output_path], mappings, method


# 8A.5 Register and Execute the Selected Document Pipeline

pipeline_result = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)


Document inspection:
{'filename': 'cloudjoi_faq.pdf', 'pages': 2, 'characters': 6177}
{'filename': 'cloudjoi_refund_policy.pdf', 'pages': 1, 'characters': 611}
{'filename': 'cloudjoi_ticketing_terms.pdf', 'pages': 5, 'characters': 18227}

Preparation completed.
FAQ records: 23
Ticketing Terms records: 17
Refund policy records: 1
Total prepared records: 41
Output: /content/week3_project/data/pipeline_v1/documents/prepared_text.jsonl
SUCCESS: documents | Inputs: 3 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json


### Test Pipeline Repeatability

In [38]:
# Cell 9 — Test Pipeline Repeatability

# 9.1 Preserve the Results from the First Run

first_run = pipeline_result


# 9.2 Execute the Same Pipeline Again

second_run = PIPELINE_RUNS["documents"]["runner"]()


# 9.3 Compare Source Fingerprints

assert (
    first_run["sources"] == second_run["sources"]
), "Source fingerprints do not match."


# 9.4 Compare Prepared Output Fingerprints

assert (
    first_run["validation"]["output_sha256"]
    == second_run["validation"]["output_sha256"]
), "Prepared output fingerprints do not match."


# 9.5 Compare Prepared Output File Sizes

assert (
    first_run["validation"]["output_bytes"]
    == second_run["validation"]["output_bytes"]
), "Prepared output sizes do not match."


# 9.6 Compare Source-to-Output Mappings

assert (
    first_run["source_output_mapping"]
    == second_run["source_output_mapping"]
), "Source-to-output mappings do not match."


# 9.7 Display Repeatability Results

print("Repeatability test: PASS")
print("Source fingerprints: MATCH")
print("Prepared output fingerprints: MATCH")
print("Prepared output sizes: MATCH")
print("Source-to-output mappings: MATCH")

Document inspection:
{'filename': 'cloudjoi_faq.pdf', 'pages': 2, 'characters': 6177}
{'filename': 'cloudjoi_refund_policy.pdf', 'pages': 1, 'characters': 611}
{'filename': 'cloudjoi_ticketing_terms.pdf', 'pages': 5, 'characters': 18227}

Preparation completed.
FAQ records: 23
Ticketing Terms records: 17
Refund policy records: 1
Total prepared records: 41
Output: /content/week3_project/data/pipeline_v1/documents/prepared_text.jsonl
SUCCESS: documents | Inputs: 3 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Repeatability test: PASS
Source fingerprints: MATCH
Prepared output fingerprints: MATCH
Prepared output sizes: MATCH
Source-to-output mappings: MATCH


### Final C1 Verification

In [39]:
# Final C1 Verification

# C1.1 Compare Complete Pipeline Validation Results

assert first_run["validation"] == second_run["validation"]


# C1.2 Verify the Prepared JSONL Output

from pathlib import Path
import json

output_path = (
    Path("/content/week3_project/data/pipeline_v1/documents")
    / "prepared_text.jsonl"
)

assert output_path.is_file()
assert output_path.stat().st_size > 0

with output_path.open("r", encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(records) == 41
assert all(record["text"].strip() for record in records)


# C1.3 Verify the Pipeline Manifest

manifest_path = output_path.parent / "pipeline_manifest.json"

assert manifest_path.is_file()

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

assert manifest["validation"] == second_run["validation"]


# C1.4 Inspect a Representative Prepared Record

print("Final C1 verification: PASS")
print("Prepared records:", len(records))
print(
    "Pipeline output count:",
    manifest["validation"]["output_count"]
)
print("Pipeline manifest: verified")

print("\nSample prepared record:")
print("Source:", records[0]["source"])
print("Text:", records[0]["text"][:400])


Final C1 verification: PASS
Prepared records: 41
Pipeline output count: 1
Pipeline manifest: verified

Sample prepared record:
Source: cloudjoi_faq.pdf
Text: Introduction
Discover everything you need to know about buying tickets, streaming live and on-demand shows, and enjoying unforgettable experiences with CloudJoi – your ultimate ticketing, streaming, and community platform in Malaysia!
